# Full-Code Notebook — Empirical MLE, FFT, Quadrature, Autodiff, Copulas, VaR / ES

This notebook is the executable companion to the merged chapter. It keeps the original workflow intact and adds autodiff, $t$-copulas, vine-copula hooks, VaR / ES, and figure-generation examples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, integrate
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.stats import norm, t, expon, weibull_min, genextreme, gaussian_kde, lognorm, multivariate_t
from numpy.polynomial.hermite import hermgauss
np.set_printoptions(precision=6, suppress=True)

## Helper functions

In [ ]:
def normalize_pdf(pdf, dx):
    pdf = np.maximum(np.asarray(pdf, dtype=float), 0.0)
    z = np.sum(pdf) * dx
    if z <= 0:
        raise ValueError('Density has non-positive integral on the grid.')
    return pdf / z

def make_grid(x_min, x_max, n_points):
    x = np.linspace(x_min, x_max, n_points)
    return x, x[1] - x[0]

def normal_pdf(x, mu, sigma):
    return norm.pdf(x, loc=mu, scale=sigma)

def student_t_pdf(x, mu, sigma, nu):
    return t.pdf(x, df=nu, loc=mu, scale=sigma)

def exponential_pdf(x, rate):
    return expon.pdf(x, loc=0.0, scale=1.0 / rate)

def convolve_independent_pdfs(component_pdfs, x, dx):
    if len(component_pdfs) == 0:
        raise ValueError('Need at least one component density.')
    pdf_sum = normalize_pdf(component_pdfs[0], dx)
    left = x[0]
    right = x[-1]
    for pdf in component_pdfs[1:]:
        pdf = normalize_pdf(pdf, dx)
        pdf_sum = signal.fftconvolve(pdf_sum, pdf, mode='full') * dx
        left += x[0]
        right += x[-1]
        pdf_sum = normalize_pdf(pdf_sum, dx)
    x_sum = np.linspace(left, right, len(pdf_sum))
    return x_sum, pdf_sum

## Independent sums: Empirical MLE + FFT

In [ ]:
class EmpiricalMLEIndependentFFT:
    def __init__(self, x_min=-10.0, x_max=10.0, grid_points=2048):
        self.x, self.dx = make_grid(x_min, x_max, grid_points)
        self.components = []
        self.data = None
        self.last_sum_grid = None
        self.last_sum_pdf = None

    def add_component(self, pdf_func, param_bounds, param_names):
        self.components.append({'pdf_func': pdf_func, 'param_bounds': list(param_bounds), 'param_names': list(param_names), 'n_params': len(param_bounds)})

    def set_data(self, data):
        self.data = np.asarray(data, dtype=float)

    def _split_params(self, params):
        params = np.asarray(params, dtype=float)
        total = sum(comp['n_params'] for comp in self.components)
        if len(params) != total:
            raise ValueError(f'Expected {total} parameters, got {len(params)}.')
        out, k = [], 0
        for comp in self.components:
            m = comp['n_params']
            out.append(params[k:k+m]); k += m
        return out

    def _sum_density(self, params):
        split = self._split_params(params)
        pdfs = [comp['pdf_func'](self.x, *p) for comp, p in zip(self.components, split)]
        return convolve_independent_pdfs(pdfs, self.x, self.dx)

    def negative_log_likelihood(self, params, floor=1e-12):
        x_sum, pdf_sum = self._sum_density(params)
        interp = interp1d(x_sum, pdf_sum, kind='linear', bounds_error=False, fill_value=0.0, assume_sorted=True)
        vals = np.maximum(interp(self.data), floor)
        self.last_sum_grid = x_sum
        self.last_sum_pdf = pdf_sum
        return float(-np.sum(np.log(vals)))

    def fit(self, initial_params, method='Nelder-Mead', options=None):
        bounds = []
        for comp in self.components:
            bounds.extend(comp['param_bounds'])
        res = minimize(self.negative_log_likelihood, x0=np.asarray(initial_params, dtype=float), method=method, bounds=bounds, options=options or {'maxiter': 5000, 'disp': True})
        self.negative_log_likelihood(res.x)
        return res

In [ ]:
# Example: Normal + Student-t + Exponential
np.random.seed(42)
n = 2500
x1 = np.random.normal(0.0, 0.8, n)
x2 = 1.2 + 0.6 * np.random.standard_t(df=4.5, size=n)
x3 = np.random.exponential(scale=1/0.7, size=n)
data_sum = x1 + x2 + x3

mle = EmpiricalMLEIndependentFFT(x_min=-12, x_max=12, grid_points=4096)
mle.add_component(normal_pdf, [(-3, 3), (0.1, 3)], ['mu_n', 'sigma_n'])
mle.add_component(student_t_pdf, [(-3, 4), (0.1, 3), (2.1, 20)], ['mu_t', 'sigma_t', 'nu_t'])
mle.add_component(exponential_pdf, [(0.05, 3.0)], ['rate_e'])
mle.set_data(data_sum)
result = mle.fit([0.2, 1.0, 0.5, 1.0, 6.0, 0.5])
result.x, result.fun

## Dependent sums: copula + quadrature

In [ ]:
def gaussian_copula_density(u, v, rho, eps=1e-12):
    u = np.clip(u, eps, 1 - eps)
    v = np.clip(v, eps, 1 - eps)
    z1 = norm.ppf(u)
    z2 = norm.ppf(v)
    denom = np.sqrt(1.0 - rho**2)
    exponent = - (z1**2 - 2.0*rho*z1*z2 + z2**2) / (2.0*(1.0 - rho**2))
    phi2 = np.exp(exponent) / (2.0*np.pi*denom)
    return phi2 / (norm.pdf(z1) * norm.pdf(z2))

def dependent_sum_pdf_quad(s, fx, Fx, fy, Fy, rho, x_lower=-np.inf, x_upper=np.inf, epsabs=1e-8, epsrel=1e-8):
    def integrand(x):
        u = Fx(x)
        y = s - x
        v = Fy(y)
        return gaussian_copula_density(u, v, rho) * fx(x) * fy(y)
    val, err = integrate.quad(integrand, x_lower, x_upper, epsabs=epsabs, epsrel=epsrel, limit=200)
    return max(val, 0.0), err

def dependent_sum_density_grid(s_grid, fx, Fx, fy, Fy, rho):
    vals, errs = [], []
    for s in s_grid:
        val, err = dependent_sum_pdf_quad(s, fx, Fx, fy, Fy, rho)
        vals.append(val); errs.append(err)
    vals = np.asarray(vals)
    vals = normalize_pdf(vals, s_grid[1]-s_grid[0])
    return vals, np.asarray(errs)

In [ ]:
rho = 0.7
fx = lambda x: t.pdf(x, df=5, loc=0.0, scale=1.0)
Fx = lambda x: t.cdf(x, df=5, loc=0.0, scale=1.0)
fy = lambda y: lognorm.pdf(y, s=0.4, scale=np.exp(0.1))
Fy = lambda y: lognorm.cdf(y, s=0.4, scale=np.exp(0.1))
s_grid = np.linspace(-6, 12, 250)
pdf_dep, err_dep = dependent_sum_density_grid(s_grid, fx, Fx, fy, Fy, rho)
np.trapz(pdf_dep, s_grid), err_dep.max()

## Factor dependence: FFT inside, Gauss–Hermite outside

In [ ]:
def gaussian_copula_conditional_pdf(x, z, rho, fx, Fx, eps=1e-12):
    u = np.clip(Fx(x), eps, 1 - eps)
    q = norm.ppf(u)
    num = norm.pdf((q - np.sqrt(rho) * z) / np.sqrt(1 - rho))
    den = np.sqrt(1 - rho) * norm.pdf(q)
    return fx(x) * num / den

def factor_dependent_sum_density_grid(s_grid, x_component_grid, component_specs, hermite_order=20):
    x = x_component_grid
    dx = x[1] - x[0]
    nodes, weights = hermgauss(hermite_order)
    accum = np.zeros_like(s_grid, dtype=float)
    for uk, wk in zip(nodes, weights):
        z = np.sqrt(2.0) * uk
        cond = [gaussian_copula_conditional_pdf(x, z, spec['rho'], spec['pdf'], spec['cdf']) for spec in component_specs]
        x_sum, pdf_sum = convolve_independent_pdfs(cond, x, dx)
        interp = interp1d(x_sum, pdf_sum, kind='linear', bounds_error=False, fill_value=0.0)
        accum += (wk / np.sqrt(np.pi)) * interp(s_grid)
    return normalize_pdf(accum, s_grid[1]-s_grid[0])

## $t$-copulas and vine-copula hook

In [ ]:
def t_copula_uniforms(rho, df, n, random_state=None):
    shape = np.array([[1.0, rho], [rho, 1.0]])
    z = multivariate_t.rvs(loc=[0.0, 0.0], shape=shape, df=df, size=n, random_state=random_state)
    return t.cdf(z[:,0], df=df), t.cdf(z[:,1], df=df)

def t_copula_sum_samples(Fx_inv, Fy_inv, rho, df, n=100000, random_state=None):
    u, v = t_copula_uniforms(rho, df, n, random_state)
    return Fx_inv(u) + Fy_inv(v)

try:
    import pyvinecopulib as pv
    HAVE_PYVINE = True
except Exception:
    HAVE_PYVINE = False
HAVE_PYVINE

## VaR / ES

In [ ]:
def empirical_var_es(losses, alpha=0.975):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    es = losses[losses >= var].mean()
    return float(var), float(es)

def var_es_from_density_grid(loss_grid, pdf_grid, alpha=0.975):
    loss_grid = np.asarray(loss_grid, dtype=float)
    pdf_grid = normalize_pdf(np.asarray(pdf_grid, dtype=float), loss_grid[1]-loss_grid[0])
    dx = loss_grid[1] - loss_grid[0]
    cdf = np.cumsum(pdf_grid) * dx
    idx = np.searchsorted(cdf, alpha)
    idx = min(max(idx, 0), len(loss_grid)-1)
    var = loss_grid[idx]
    mask = loss_grid >= var
    tail_prob = np.sum(pdf_grid[mask]) * dx
    es = np.sum(loss_grid[mask]*pdf_grid[mask]) * dx / tail_prob
    return float(var), float(es)

In [ ]:
Fx_inv = lambda u: t.ppf(u, df=5, loc=0.0, scale=1.0)
Fy_inv = lambda u: lognorm.ppf(u, s=0.4, scale=np.exp(0.1))
sum_samples = t_copula_sum_samples(Fx_inv, Fy_inv, rho=0.7, df=4, n=30000, random_state=2026)
empirical_var_es(sum_samples, alpha=0.975)

## Visualizations

In [ ]:
# Gaussian vs t-copula tail behavior
np.random.seed(123)
n = 3000
rho = 0.7
cov = np.array([[1.0, rho], [rho, 1.0]])
z = np.random.multivariate_normal([0, 0], cov, size=n)
ug, vg = norm.cdf(z[:,0]), norm.cdf(z[:,1])
xt = multivariate_t.rvs(loc=[0,0], shape=cov, df=4, size=n, random_state=123)
ut, vt = t.cdf(xt[:,0], df=4), t.cdf(xt[:,1], df=4)
fig, axes = plt.subplots(1,2, figsize=(11,5))
axes[0].scatter(ug, vg, s=6, alpha=0.35)
axes[0].set_title('Gaussian copula')
axes[1].scatter(ut, vt, s=6, alpha=0.35, color='darkorange')
axes[1].set_title('$t$-copula ($\nu=4$)')
for ax in axes:
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_xlabel('$u$'); ax.set_ylabel('$v$')
    ax.axvline(0.95, ls='--', c='r', lw=1); ax.axhline(0.95, ls='--', c='r', lw=1)
plt.suptitle('Same correlation, different tail behavior')
plt.show()

In [ ]:
# Loss distribution with VaR and ES
np.random.seed(321)
losses = 0.7*np.random.normal(0,1,8000) + 0.3*np.random.standard_t(df=4, size=8000)*1.4
var, es = empirical_var_es(losses, alpha=0.975)
plt.figure(figsize=(10,5))
plt.hist(losses, bins=120, density=True, alpha=0.45, color='steelblue')
plt.axvline(var, color='crimson', lw=2, ls='--', label=f'VaR={var:.2f}')
plt.axvline(es, color='darkgreen', lw=2, ls='-.', label=f'ES={es:.2f}')
plt.legend(); plt.xlabel('Loss'); plt.ylabel('Density'); plt.title('VaR and ES')
plt.show()

## Optional JAX autodiff demo

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    from jax import grad
    HAVE_JAX = True
except Exception:
    HAVE_JAX = False
print('JAX available:', HAVE_JAX)
if HAVE_JAX:
    def normal_pdf_jax(x, mu, sigma):
        return (1.0 / (jnp.sqrt(2.0*jnp.pi) * sigma)) * jnp.exp(-0.5*((x-mu)/sigma)**2)
    def fft_convolve_two_pdfs_jax(pdf1, pdf2, dx):
        n = pdf1.shape[0]
        pad = n - 1
        a = jnp.pad(pdf1, (0, pad))
        b = jnp.pad(pdf2, (0, pad))
        return jnp.fft.ifft(jnp.fft.fft(a) * jnp.fft.fft(b)).real * dx
    def negative_loglik_fft_jax(params, data, x_grid):
        mu1, sigma1, mu2, sigma2 = params
        dx = x_grid[1] - x_grid[0]
        pdf1 = normal_pdf_jax(x_grid, mu1, sigma1); pdf1 = pdf1 / (jnp.sum(pdf1)*dx)
        pdf2 = normal_pdf_jax(x_grid, mu2, sigma2); pdf2 = pdf2 / (jnp.sum(pdf2)*dx)
        pdf_sum = fft_convolve_two_pdfs_jax(pdf1, pdf2, dx)
        x_sum = jnp.linspace(x_grid[0]+x_grid[0], x_grid[-1]+x_grid[-1], pdf_sum.shape[0])
        idx = jnp.clip(jnp.searchsorted(x_sum, data, side='right') - 1, 0, len(x_sum)-2)
        x0, x1 = x_sum[idx], x_sum[idx+1]
        y0, y1 = pdf_sum[idx], pdf_sum[idx+1]
        w = (data - x0) / (x1 - x0)
        vals = jnp.maximum((1-w)*y0 + w*y1, 1e-12)
        return -jnp.sum(jnp.log(vals))
    x_grid = jnp.linspace(-8, 8, 1024)
    data = jnp.array(np.random.normal(size=200))
    params = jnp.array([0.0, 1.0, 0.2, 1.1])
    print(grad(negative_loglik_fft_jax)(params, data, x_grid))